# Specimen 02 — Planner → Sequential Workers

Goal: one agent decomposes a task into ordered subtasks; a worker agent executes each one in turn, carrying real state forward between steps.

In [1]:
import os
import json
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.environ['ANTHROPIC_API_KEY'])
MODEL = 'claude-opus-5'

INPUT_PRICE_PER_MTOK = 5.00
OUTPUT_PRICE_PER_MTOK = 25.00

def call_cost(usage):
    return (usage.input_tokens / 1_000_000 * INPUT_PRICE_PER_MTOK) + (usage.output_tokens / 1_000_000 * OUTPUT_PRICE_PER_MTOK)

def call_model(messages, tools=None, max_tokens=800, output_schema=None):
    kwargs = dict(model=MODEL, max_tokens=max_tokens, messages=messages, thinking={"type": "disabled"})
    if tools:
        kwargs['tools'] = tools
    if output_schema:
        kwargs['output_config'] = {"format": {"type": "json_schema", "schema": output_schema}}
    return client.messages.create(**kwargs)


`call_model` bakes in two things learned the hard way in Phase 3/4: `thinking` is disabled by default, since `claude-opus-5` defaults to extended thinking and can silently burn a small `max_tokens` budget before producing a visible tool call or text block; and `output_schema` makes structured JSON handoffs a one-liner, since Phase 3's chain-of-thought grading bug came from regex-parsing freeform prose instead of forcing a schema. Every agent-to-agent handoff in this phase should go through `output_schema`, not string parsing.

## 1. Define the planner agent

Given a broad task, it outputs an ordered list of subtasks as structured JSON (a schema with a `subtasks: [{step, description}]` shape) — not a numbered list you regex-split out of prose.

In [2]:
PLANNER_SYSTEM = (
    "You are a planning agent. Given a broad task, break it into an ordered list of concrete, "
    "sequential subtasks. Each subtask must be something a worker agent can complete using only "
    "its own reasoning plus the results of prior subtasks -- no external tools or live data."
)

PLAN_SCHEMA = {
    "type": "object",
    "properties": {
        "subtasks": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "step": {"type": "integer"},
                    "description": {"type": "string"},
                },
                "required": ["step", "description"],
                "additionalProperties": False,
            },
        },
    },
    "required": ["subtasks"],
    "additionalProperties": False,
}

def run_planner(task, max_subtasks=6):
    response = call_model(
        messages=[{"role": "user", "content": (
            f"{PLANNER_SYSTEM}\n\nTask: {task}\n\nBreak this into at most {max_subtasks} ordered subtasks."
        )}],
        output_schema=PLAN_SCHEMA,
        max_tokens=2000,
    )
    text = ''.join(b.text for b in response.content if b.type == 'text')
    plan = json.loads(text)
    return plan["subtasks"], response.usage

subtasks, usage = run_planner("Plan a 3-day trip to Kyoto, Japan, then produce a day-by-day itinerary with a total estimated budget in USD.")
for st in subtasks:
    print(f"{st['step']}. {st['description']}")

1. Define the trip parameters and assumptions from general knowledge: traveler profile (e.g., 2 adults, mid-range budget), best-guess season/dates and typical weather, arrival/departure times and gateway (e.g., arrive Kyoto Station via Shinkansen from Osaka/Tokyo), trip pace, and interests (temples, gardens, food, culture). State currency assumption (USD with an assumed JPY exchange rate). Output a concise assumptions list that all later steps must use.
2. From knowledge, compile a shortlist of Kyoto attractions, neighborhoods, and experiences (e.g., Fushimi Inari, Kiyomizu-dera, Gion/Higashiyama, Arashiyama bamboo grove, Kinkaku-ji, Ryoan-ji, Nishiki Market, Philosopher's Path, Nijo Castle) with for each: district/location, approximate visit duration, typical opening hours, and admission fee in JPY and USD. Note 1-2 optional day-trip or evening options.
3. Group the shortlisted sights into three geographically coherent day clusters (e.g., Day 1 Southern Higashiyama + Gion, Day 2 Arash

## 2. Define the worker agent

Executes exactly one subtask, with the previous subtasks' outputs available as context.

In [3]:
WORKER_SYSTEM = (
    "You are a worker agent. You execute exactly one subtask at a time. You are given the current "
    "subtask plus the results of all previous subtasks as context. Use the prior results directly -- "
    "do not re-derive information that was already given to you."
)

def run_worker(subtask_description, prior_results):
    context = "\n".join(f"Step {i+1} result: {r}" for i, r in enumerate(prior_results)) if prior_results else "(no prior results yet)"
    response = call_model(
        messages=[{"role": "user", "content": (
            f"{WORKER_SYSTEM}\n\nPrior results:\n{context}\n\nCurrent subtask: {subtask_description}\n\n"
            "Produce the result for this subtask only."
        )}],
        max_tokens=500,
    )
    text = ''.join(b.text for b in response.content if b.type == 'text')
    return text, response.usage

first_result, usage = run_worker(subtasks[0]["description"], [])
print(first_result)

# Trip Parameters & Assumptions — Kyoto 4-Day Itinerary

All later steps must use these fixed assumptions.

## 1. Traveler Profile
- **Party:** 2 adults traveling together (couple or two friends), ages ~30–45
- **Fitness/mobility:** Good — comfortable walking 12,000–18,000 steps/day, some stairs and gentle hill climbs (e.g., Fushimi Inari lower loop, Kiyomizu-dera approach)
- **Experience level:** First-time visitors to Kyoto; may have visited Tokyo/Osaka before
- **Language:** English-speaking, no Japanese; assume reliance on English signage, translation apps, and pointing-at-menus
- **Budget tier:** Mid-range ("comfortable but not luxury") — 3-star to 4-star hotels or a nice ryokan splurge for one night, mix of casual and one or two special meals
- **Travel style:** Independent (no group tours), willing to book 1–2 timed reservations/activities in advance

## 2. Season & Dates (best guess)
- **Assumed dates:** Early-to-mid **November** (peak autumn foliage season) — e.g., **Nov 8–12*

## 3. Run the plan sequentially

Loop through the planner's subtasks in order, feeding each worker's output into the next step's context. This loop is deliberately a plain `for`, not `asyncio` — that's the whole point of Phase 5 being sequential first.

In [4]:
def run_plan(task, max_subtasks=6):
    subtasks, planner_usage = run_planner(task, max_subtasks=max_subtasks)
    results = []
    usages = [planner_usage]
    for st in subtasks:
        result, usage = run_worker(st["description"], results)
        results.append(result)
        usages.append(usage)
    return subtasks, results, usages

TASK = "Plan a 3-day trip to Kyoto, Japan, then produce a day-by-day itinerary with a total estimated budget in USD."
subtasks, results, usages = run_plan(TASK)
for st, r in zip(subtasks, results):
    print(f"Step {st['step']}: {st['description']}")
    print(f"  -> {r}\n")

Step 1: Define the trip parameters and assumptions: travel season/dates (assume a specific month, e.g. late October), traveler profile (e.g. two adults, mid-range budget), arrival/departure times and entry point (e.g. arrive Kyoto Station from Kansai Airport), pace preferences, and interests (temples, gardens, food, culture). State all assumptions explicitly since no live data is available.
  -> # Trip Parameters & Assumptions — Kyoto Itinerary Planning

*All parameters below are explicit planning assumptions. No live data (schedules, prices, availability, weather, closures, event calendars) was consulted; every figure is an estimate based on general, non-real-time knowledge and should be re-verified against official sources before booking.*

---

## 1. Travel Season & Dates

| Parameter | Assumption |
|---|---|
| **Season** | Late autumn (*kōyō* / fall foliage season) |
| **Assumed dates** | **Saturday 25 October – Wednesday 29 October** (year unspecified; treat as a generic late-Octo

## 4. Design a task where step 2 truly depends on step 1

Pick a task where the second subtask can't be done correctly without the specific output of the first — the worker should visibly fail or go generic if you strip the earlier context out. Confirm state is actually flowing, not just being ignored.

In [5]:
verify_idx = 1 if len(subtasks) > 1 else 0
verify_subtask = subtasks[verify_idx]["description"]

with_context, _ = run_worker(verify_subtask, results[:verify_idx])
without_context, _ = run_worker(verify_subtask, [])

print("WITH prior context:\n", with_context)
print("\nWITHOUT prior context:\n", without_context)
print("\nResults differ meaningfully:", with_context.strip() != without_context.strip())

WITH prior context:
 # Kyoto Candidate Attractions — Master List by District

**⚠️ Data caveat:** All hours, fees, and durations below are **estimates from general knowledge**, not live-verified. Japanese temple admission fees are raised periodically and several major sites revised prices in recent years. Seasonal "special openings" (*tokubetsu kōkai*) and autumn night illuminations (*yoru no tokubetsu haikan*) change annually. **Re-verify every entry against official websites before finalizing the itinerary.**

**Notation:** Duration = time on-site, excluding transit. "Best time" reflects crowd patterns, light quality, and the late-October constraint of **~17:00 effective daylight end**.

---

## DISTRICT 1 — Southern Higashiyama / Gion
*The classic "old Kyoto" walking corridor. Highest tourist density in the city. Most sites are within walking distance of each other along a roughly north–south spine.*

| # | Site | Typical Hours | Duration | Fee (JPY) | Best Time of Day |
|---|---|--

## 5. Cap the planner's own subtask count

Reject or truncate a plan with more than N subtasks. This is a planner-side guardrail, distinct from a worker-loop step cap — the planner itself is a place things can run away.

In [6]:
def run_planner_capped(task, max_subtasks=5):
    subtasks, usage = run_planner(task, max_subtasks=max_subtasks)
    if len(subtasks) > max_subtasks:
        print(f"Planner returned {len(subtasks)} subtasks (over the cap of {max_subtasks}) -- truncating.")
        subtasks = subtasks[:max_subtasks]
    return subtasks, usage

oversized_task = "Write an extremely detailed, comprehensive guide to becoming a professional chef, broken into as many granular subtasks as make sense."
capped_subtasks, _ = run_planner_capped(oversized_task, max_subtasks=5)
print(f"Final subtask count: {len(capped_subtasks)} (cap was 5)")
for st in capped_subtasks:
    print(f"  {st['step']}: {st['description']}")

Final subtask count: 5 (cap was 5)
  1: Define the guide's scope, audience, and architecture. Specify the target readers (career-changers, high-school graduates, home cooks, hospitality workers), the geographic/industry contexts covered (restaurants, hotels, private/personal cheffing, catering, R&D), and the end-state definitions of 'professional chef' (commis through executive chef). Produce a complete annotated table of contents with every chapter and sub-section, word-count targets per section, a glossary plan, and a list of recurring structural elements (checklists, timelines, sample schedules, case-study vignettes, self-assessment rubrics). This outline governs all later steps.
  2: Write the foundational sections: (a) the reality of the profession — kitchen brigade hierarchy and role-by-role duties, daily/weekly life, hours, pay ranges by role and venue type, physical and psychological demands, career risks and burnout, and honest self-assessment questionnaires; (b) the education

## 6. Track cost across the whole plan execution

Sum cost across the planner call plus every worker call in the run.

In [7]:
subtasks, results, usages = run_plan(TASK)
total_cost = sum(call_cost(u) for u in usages)
print(f"Subtasks executed: {len(subtasks)}")
print(f"Total cost across planner + all workers: ${total_cost:.6f}")

Subtasks executed: 6
Total cost across planner + all workers: $0.149820
